# DDI Network Visualization — Gephi Data Preparation (v2)

**Purpose:** Generate node and edge list files for Gephi using WHO ATC classification for drug class coloring.

**Input:**
- `FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv` — DDI signal data from the main analysis
- `drug_atc_mapping.csv` — ATC classification from `ATC_Drug_Classification.ipynb`

**Output:**
- `gephi_nodes.csv` / `gephi_edges.csv` — full network
- `gephi_nodes_top100.csv` / `gephi_edges_top100.csv` — main paper figure
- `gephi_nodes_top50.csv` / `gephi_edges_top50.csv` — clean figure
- `gephi_class_network_nodes.csv` / `gephi_class_network_edges.csv` — class-level aggregated network

**Gephi Import Instructions:**
1. Open Gephi → File → New Project
2. File → Import Spreadsheet → `gephi_edges_top100.csv` → Edge Table
3. File → Import Spreadsheet → `gephi_nodes_top100.csv` → Node Table → Append
4. Filters → Topology → Degree Range → set min to 2 (removes isolated nodes)
5. Layout → ForceAtlas2 → Scaling: 100, Gravity: 1.0, check Prevent Overlap → Run ~30s → Stop
6. Appearance → Nodes → Size → Ranking → FREQ → Min: 20, Max: 80 → Apply
7. Appearance → Nodes → Color → Partition → RISK_CLASS → Apply
8. Bottom of graph → click "T" for labels → adjust size slider
9. Preview → Refresh → Export as SVG or PNG (300 DPI)


In [45]:
import pandas as pd
import numpy as np
from collections import Counter

print("Libraries loaded")


Libraries loaded


## Configuration

In [46]:
# ---- CONFIGURATION ----
ROR_THRESHOLD = 2.0        # Minimum ROR for inclusion
MIN_SAMPLE_SIZE = 50       # Minimum N_EXPOSED for inclusion
MAX_EDGES = 200            # Top N edges by ROR

print(f"ROR threshold: {ROR_THRESHOLD}")
print(f"Min sample size: {MIN_SAMPLE_SIZE}")
print(f"Max edges: {MAX_EDGES}")


ROR threshold: 2.0
Min sample size: 50
Max edges: 200


## Load Signal Data and ATC Mapping

In [47]:
# ---- LOAD SIGNAL DATA ----
try:
    signals = pd.read_csv("FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv")
    print(f"Loaded high-confidence signals: {len(signals):,}")
except FileNotFoundError:
    signals = pd.read_csv("FAERS_DDI_SIGNALS.csv")
    print(f"Loaded all signals: {len(signals):,}")

# ---- LOAD ATC MAPPING ----
atc_map = pd.read_csv("drug_atc_mapping.csv")
print(f"Loaded ATC mapping: {len(atc_map)} drugs")

# Build lookup dictionaries
drug_to_risk_class = dict(zip(atc_map['DRUG'], atc_map['RISK_CLASS']))
drug_to_atc_l1 = dict(zip(atc_map['DRUG'], atc_map['ATC_L1_NAME']))
drug_to_atc_l2 = dict(zip(atc_map['DRUG'], atc_map['ATC_L2_CODE']))

classified = sum(1 for v in drug_to_risk_class.values() if v != 'Unclassified')
print(f"Classified drugs: {classified}/{len(atc_map)} ({classified/len(atc_map)*100:.1f}%)")

# ---- FILTER SIGNALS ----
print(f"\nFiltering: ROR > {ROR_THRESHOLD}, N >= {MIN_SAMPLE_SIZE}")
filtered = signals[
    (signals['ROR'] > ROR_THRESHOLD) &
    (signals['N_EXPOSED'] >= MIN_SAMPLE_SIZE)
].copy()

print(f"After filtering: {len(filtered):,} signals")

filtered = filtered.nlargest(MAX_EDGES, 'ROR')
print(f"Top {MAX_EDGES} edges selected")


Loaded high-confidence signals: 5,453
Loaded ATC mapping: 684 drugs
Classified drugs: 662/684 (96.8%)

Filtering: ROR > 2.0, N >= 50
After filtering: 5,453 signals
Top 200 edges selected


## Create Edge List

In [48]:
# ---- CREATE EDGE LIST ----
edges = []
for _, row in filtered.iterrows():
    pair = row['PAIR']
    drugs = pair.split(' + ')
    if len(drugs) == 2:
        edges.append({
            'Source': drugs[0].strip(),
            'Target': drugs[1].strip(),
            'Weight': round(row['ROR'], 2),
            'ROR': round(row['ROR'], 2),
            'N_EXPOSED': int(row['N_EXPOSED']),
            'A_SERIOUS': int(row['A_SERIOUS_EXPOSED']),
            'Type': 'Undirected',
            'Label': pair
        })

edges_df = pd.DataFrame(edges)

# ROR severity categories
edges_df['SEVERITY'] = pd.cut(
    edges_df['ROR'],
    bins=[0, 3, 5, 10, float('inf')],
    labels=['MODERATE (2-3)', 'STRONG (3-5)', 'VERY_STRONG (5-10)', 'EXTREME (>10)']
)

print(f"Edges created: {len(edges_df):,}")
print(f"ROR range: {edges_df['ROR'].min():.1f} - {edges_df['ROR'].max():.1f}")
print(f"\nEdges by severity:")
print(edges_df['SEVERITY'].value_counts())


Edges created: 200
ROR range: 6.8 - 8.7

Edges by severity:
SEVERITY
VERY_STRONG (5-10)    200
MODERATE (2-3)          0
STRONG (3-5)            0
EXTREME (>10)           0
Name: count, dtype: int64


## Create Node List with ATC Classification

Each node gets its therapeutic class from the ATC mapping file, replacing the old manual dictionary approach.

In [49]:
# ---- CREATE NODE LIST ----
all_drugs = edges_df['Source'].tolist() + edges_df['Target'].tolist()
drug_freq = Counter(all_drugs)

nodes = []
for drug, freq in drug_freq.items():
    risk_class = drug_to_risk_class.get(drug, 'Unclassified')
    atc_l1 = drug_to_atc_l1.get(drug, '')
    atc_l2 = drug_to_atc_l2.get(drug, '')

    # Calculate ROR stats for this drug
    drug_edges = edges_df[(edges_df['Source'] == drug) | (edges_df['Target'] == drug)]
    avg_ror = drug_edges['ROR'].mean() if len(drug_edges) > 0 else 0
    max_ror = drug_edges['ROR'].max() if len(drug_edges) > 0 else 0

    nodes.append({
        'Id': drug,
        'Label': drug,
        'FREQ': freq,
        'RISK_CLASS': risk_class,       # For Gephi color (Partition)
        'ATC_L1': atc_l1,              # Broad anatomical group
        'ATC_L2': atc_l2,              # Therapeutic subgroup code
        'AVG_ROR': round(avg_ror, 2),
        'MAX_ROR': round(max_ror, 2),
    })

nodes_df = pd.DataFrame(nodes)

print(f"Nodes created: {len(nodes_df):,}")
print(f"\nDrugs by risk class (in network):")
print(nodes_df['RISK_CLASS'].value_counts().to_string())

# How many are unclassified?
n_unclass = (nodes_df['RISK_CLASS'] == 'Unclassified').sum()
print(f"\nUnclassified in network: {n_unclass}/{len(nodes_df)} ({n_unclass/len(nodes_df)*100:.1f}%)")

print(f"\nTop 20 most connected drugs:")
print(nodes_df.nlargest(20, 'FREQ')[['Label', 'FREQ', 'RISK_CLASS', 'AVG_ROR']].to_string(index=False))


Nodes created: 129

Drugs by risk class (in network):
RISK_CLASS
Immunosuppressant            25
Alimentary/Metabolism        14
Analgesic/Opioid             11
Antineoplastic               10
Corticosteroid                8
Antibacterial                 8
Psycholeptic                  6
Anti-inflammatory (NSAID)     6
Cardiovascular (Other)        4
Musculoskeletal (Other)       4
Respiratory (Other)           4
RAAS Inhibitor                3
Unclassified                  3
Antiparasitic                 3
Anti-infective (Other)        2
Blood (Other)                 2
Various                       2
Antiepileptic                 2
Anesthetic                    2
Antidiabetic                  2
Antithrombotic                2
Dermatological                2
Lipid Modifying               1
Psychoanaleptic               1
Antiviral                     1
Nervous System (Other)        1

Unclassified in network: 3/129 (2.3%)

Top 20 most connected drugs:
                                  

## Save Drug-Level Gephi Files

In [50]:
# ---- FULL NETWORK ----
nodes_df.to_csv("gephi_nodes.csv", index=False)
edges_df.to_csv("gephi_edges.csv", index=False)
print(f"Full network: {len(nodes_df)} nodes, {len(edges_df)} edges")

# ---- TOP 100 (MAIN FIGURE) ----
top100 = edges_df.nlargest(100, 'ROR')
top100_drugs = set(top100['Source'].tolist() + top100['Target'].tolist())
top100_nodes = nodes_df[nodes_df['Id'].isin(top100_drugs)]

top100_nodes.to_csv("gephi_nodes_top100.csv", index=False)
top100.to_csv("gephi_edges_top100.csv", index=False)
print(f"Top 100: {len(top100_nodes)} nodes, 100 edges")

# ---- TOP 50 (CLEAN FIGURE) ----
top50 = edges_df.nlargest(50, 'ROR')
top50_drugs = set(top50['Source'].tolist() + top50['Target'].tolist())
top50_nodes = nodes_df[nodes_df['Id'].isin(top50_drugs)]

top50_nodes.to_csv("gephi_nodes_top50.csv", index=False)
top50.to_csv("gephi_edges_top50.csv", index=False)
print(f"Top 50:  {len(top50_nodes)} nodes, 50 edges")


Full network: 129 nodes, 200 edges
Top 100: 87 nodes, 100 edges
Top 50:  62 nodes, 50 edges


## Class-Level Aggregated Network

This collapses individual drugs into their therapeutic classes, creating a cleaner "big picture" graph. Each node is a drug class (e.g., "Antineoplastic"), and each edge represents the number of DDI signals between those two classes, weighted by average ROR.

This is often more publishable than the drug-level graph because it tells a clearer story without overwhelming the reader with hundreds of drug names.


In [51]:
# ---- BUILD CLASS-LEVEL NETWORK ----
class_edges = {}

for _, row in edges_df.iterrows():
    src_class = drug_to_risk_class.get(row['Source'], 'Unclassified')
    tgt_class = drug_to_risk_class.get(row['Target'], 'Unclassified')

    # Skip self-loops (both drugs in same class)
    # and skip Unclassified
    if src_class == tgt_class:
        continue
    if src_class == 'Unclassified' or tgt_class == 'Unclassified':
        continue

    # Normalize edge key (alphabetical order)
    key = tuple(sorted([src_class, tgt_class]))

    if key not in class_edges:
        class_edges[key] = {'count': 0, 'ror_sum': 0, 'max_ror': 0, 'pairs': []}

    class_edges[key]['count'] += 1
    class_edges[key]['ror_sum'] += row['ROR']
    class_edges[key]['max_ror'] = max(class_edges[key]['max_ror'], row['ROR'])
    class_edges[key]['pairs'].append(row['Label'])

# Build edge table
class_edge_rows = []
for (src, tgt), stats in class_edges.items():
    class_edge_rows.append({
        'Source': src,
        'Target': tgt,
        'Weight': stats['count'],              # Number of DDI signals between classes
        'DDI_COUNT': stats['count'],
        'AVG_ROR': round(stats['ror_sum'] / stats['count'], 2),
        'MAX_ROR': round(stats['max_ror'], 2),
        'Type': 'Undirected',
        'Label': f"{src} <-> {tgt}"
    })

class_edges_df = pd.DataFrame(class_edge_rows).sort_values('DDI_COUNT', ascending=False)

# Build node table
class_node_counts = Counter()
for _, row in edges_df.iterrows():
    src_class = drug_to_risk_class.get(row['Source'], 'Unclassified')
    tgt_class = drug_to_risk_class.get(row['Target'], 'Unclassified')
    if src_class != 'Unclassified':
        class_node_counts[src_class] += 1
    if tgt_class != 'Unclassified':
        class_node_counts[tgt_class] += 1

class_node_rows = []
for cls, count in class_node_counts.items():
    n_drugs = len(nodes_df[nodes_df['RISK_CLASS'] == cls])
    class_node_rows.append({
        'Id': cls,
        'Label': cls,
        'FREQ': count,           # Total DDI signal involvement
        'NUM_DRUGS': n_drugs,    # How many individual drugs in this class
    })

class_nodes_df = pd.DataFrame(class_node_rows).sort_values('FREQ', ascending=False)

print("CLASS-LEVEL NETWORK")
print("=" * 60)
print(f"Nodes (drug classes): {len(class_nodes_df)}")
print(f"Edges (class-class interactions): {len(class_edges_df)}")

print(f"\nTop 15 class-class interaction pairs (by DDI signal count):")
for _, row in class_edges_df.head(15).iterrows():
    print(f"  {row['Source']:30s} <-> {row['Target']:30s}  signals: {row['DDI_COUNT']:3d}  avg ROR: {row['AVG_ROR']:.1f}")

print(f"\nClass node sizes:")
for _, row in class_nodes_df.iterrows():
    print(f"  {row['Label']:30s}  involvement: {row['FREQ']:4d}  drugs: {row['NUM_DRUGS']:3d}")


CLASS-LEVEL NETWORK
Nodes (drug classes): 25
Edges (class-class interactions): 69

Top 15 class-class interaction pairs (by DDI signal count):
  Immunosuppressant              <-> Respiratory (Other)             signals:  16  avg ROR: 8.0
  Alimentary/Metabolism          <-> Immunosuppressant               signals:  14  avg ROR: 7.7
  Corticosteroid                 <-> Immunosuppressant               signals:   9  avg ROR: 7.5
  Anti-inflammatory (NSAID)      <-> Immunosuppressant               signals:   8  avg ROR: 7.7
  Dermatological                 <-> Immunosuppressant               signals:   6  avg ROR: 8.0
  Alimentary/Metabolism          <-> Analgesic/Opioid                signals:   6  avg ROR: 7.7
  Immunosuppressant              <-> RAAS Inhibitor                  signals:   6  avg ROR: 7.9
  Analgesic/Opioid               <-> Immunosuppressant               signals:   5  avg ROR: 7.9
  Antibacterial                  <-> Immunosuppressant               signals:   4  avg RO

## Save Class-Level Gephi Files

In [52]:
# ---- SAVE CLASS-LEVEL NETWORK ----
class_nodes_df.to_csv("gephi_class_network_nodes.csv", index=False)
class_edges_df.to_csv("gephi_class_network_edges.csv", index=False)
print(f"Saved: gephi_class_network_nodes.csv ({len(class_nodes_df)} class nodes)")
print(f"Saved: gephi_class_network_edges.csv ({len(class_edges_df)} class edges)")

print(f"\n{'=' * 60}")
print("GEPHI DATA PREPARATION COMPLETE (v2 — ATC-based)")
print(f"{'=' * 60}")
print(f"\nDrug-level files:")
print(f"  gephi_nodes.csv + gephi_edges.csv              (full network)")
print(f"  gephi_nodes_top100.csv + gephi_edges_top100.csv (main figure)")
print(f"  gephi_nodes_top50.csv + gephi_edges_top50.csv   (clean figure)")
print(f"\nClass-level files:")
print(f"  gephi_class_network_nodes.csv + gephi_class_network_edges.csv")
print(f"\nGephi tips:")
print(f"  Drug-level:  Color by RISK_CLASS (Partition), Size by FREQ")
print(f"  Class-level: Color by Label (Partition), Size by NUM_DRUGS or FREQ")
print(f"               Edge thickness by DDI_COUNT, Edge color by AVG_ROR")


Saved: gephi_class_network_nodes.csv (25 class nodes)
Saved: gephi_class_network_edges.csv (69 class edges)

GEPHI DATA PREPARATION COMPLETE (v2 — ATC-based)

Drug-level files:
  gephi_nodes.csv + gephi_edges.csv              (full network)
  gephi_nodes_top100.csv + gephi_edges_top100.csv (main figure)
  gephi_nodes_top50.csv + gephi_edges_top50.csv   (clean figure)

Class-level files:
  gephi_class_network_nodes.csv + gephi_class_network_edges.csv

Gephi tips:
  Drug-level:  Color by RISK_CLASS (Partition), Size by FREQ
  Class-level: Color by Label (Partition), Size by NUM_DRUGS or FREQ
               Edge thickness by DDI_COUNT, Edge color by AVG_ROR


In [53]:
import pandas as pd

COLOR_MAP = {
    'Immunosuppressant': (230, 57, 70),
    'Anti-inflammatory (NSAID)': (231, 111, 81),
    'Corticosteroid': (244, 162, 97),
    'Analgesic/Opioid': (38, 70, 83),
    'Antineoplastic': (42, 157, 143),
    'Antithrombotic': (69, 123, 157),
    'Alimentary/Metabolism': (138, 177, 125),
    'Antibacterial': (106, 153, 78),
    'Anti-infective (Other)': (56, 102, 65),
    'Antiviral': (77, 144, 142),
    'Respiratory (Other)': (144, 190, 109),
    'Cardiovascular (Other)': (87, 117, 144),
    'RAAS Inhibitor': (67, 97, 160),
    'Lipid Modifying': (123, 151, 212),
    'Psycholeptic': (155, 93, 229),
    'Psychoanaleptic': (177, 133, 219),
    'Antiepileptic': (201, 173, 229),
    'Nervous System (Other)': (212, 191, 239),
    'Dermatological': (188, 108, 37),
    'Musculoskeletal (Other)': (221, 161, 94),
    'Antidiabetic': (96, 108, 56),
    'Blood (Other)': (61, 90, 128),
    'Anesthetic': (119, 141, 169),
    'Antiparasitic': (163, 177, 138),
    'Various': (153, 153, 153),
    'Unclassified': (204, 204, 204),
}

nodes = pd.read_csv("gephi_class_network_nodes.csv")

# Drop old color column if it exists
if 'color' in nodes.columns:
    nodes = nodes.drop(columns=['color'])

# Add r, g, b columns
nodes['r'] = nodes['Label'].map(lambda x: COLOR_MAP.get(x, (204,204,204))[0])
nodes['g'] = nodes['Label'].map(lambda x: COLOR_MAP.get(x, (204,204,204))[1])
nodes['b'] = nodes['Label'].map(lambda x: COLOR_MAP.get(x, (204,204,204))[2])

nodes.to_csv("gephi_class_network_nodes.csv", index=False)
print("RGB columns added. Re-import into Gephi.")
print(nodes[['Label', 'r', 'g', 'b']].to_string())

nodes = pd.read_csv("gephi_class_network_nodes.csv")
nodes['drug_class'] = nodes['Label']
nodes.to_csv("gephi_class_network_nodes.csv", index=False)
print("Done - drug_class column added")

RGB columns added. Re-import into Gephi.
                        Label    r    g    b
0           Immunosuppressant  230   57   70
1       Alimentary/Metabolism  138  177  125
2            Analgesic/Opioid   38   70   83
3   Anti-inflammatory (NSAID)  231  111   81
4         Respiratory (Other)  144  190  109
5              Corticosteroid  244  162   97
6              Antineoplastic   42  157  143
7     Musculoskeletal (Other)  221  161   94
8               Antibacterial  106  153   78
9              Dermatological  188  108   37
10               Psycholeptic  155   93  229
11             RAAS Inhibitor   67   97  160
12              Antiparasitic  163  177  138
13              Blood (Other)   61   90  128
14     Cardiovascular (Other)   87  117  144
15               Antidiabetic   96  108   56
16             Antithrombotic   69  123  157
17              Antiepileptic  201  173  229
18                    Various  153  153  153
19                 Anesthetic  119  141  169
20     Anti-in